# [M3] XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
## Buổi S01: PyTorch Tensors, Autograd & Đồ Thị Tính Toán Tự Động

---

### Metadata Khóa Học
* **Module:** M3 — XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
* **Buổi:** S01
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook lập trình tối ưu hóa đạo hàm tự động bằng PyTorch Autograd
* **Mức độ liên quan VOAI / Olympic AI:** Làm chủ tầng tính toán vi phân của PyTorch, sẵn sàng viết Custom Loss cho phòng thi.

#### Mục tiêu học tập (Learning Objectives)
- Làm chủ cấu trúc dữ liệu nền tảng PyTorch Tensor và sự tương đồng/khác biệt với NumPy ndarray.
- Hiểu cơ chế quản lý thiết bị tính toán: Chuyển đổi linh hoạt giữa CPU và GPU (CUDA/MPS).
- Nắm vững cơ chế Autograd (Tự động tính đạo hàm) và Đồ thị tính toán (Computational Graph).
- Tự tay lập trình thuật toán Hồi quy tuyến tính từ số 0 chỉ bằng PyTorch Tensors và cơ chế lan truyền ngược (`.backward()`).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Module M2 (Đặc biệt là M2_S02 về Hồi quy Tuyến tính và hàm mất mát).
- Kiến thức đại số tuyến tính: Phép nhân ma trận và đạo hàm hàm hợp (Chain Rule).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong Machine Learning cổ điển (Scikit-Learn), bạn chỉ cần gọi `model.fit()`. Mọi phép toán đạo hàm tối ưu đều bị giấu kín trong một "hộp đen" (Black-box). Nhưng khi bước vào Học Sâu (Deep Learning) và thi đấu Olympic AI, các kiến trúc mạng nơ-ron có thể có hàng chục triệu tham số với các kết nối phi tuyến phức tạp, đòi hỏi bạn phải can thiệp trực tiếp vào từng tầng tính toán.

PyTorch không chỉ là một thư viện Deep Learning; nó là một **Nền tảng Tính toán Ma trận Đa chiều có Tự động Tính Đạo hàm (Automatic Differentiation Engine - Autograd) được tăng tốc bằng GPU**. Hiểu cách PyTorch theo dõi từng phép toán để tự động tính gradient là chìa khóa mở toang cánh cửa bước vào kỷ nguyên AI hiện đại!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. PyTorch Tensor vs NumPy ndarray
* Tương đồng: Đều là mảng đa chiều hỗ trợ tính toán số học vectơ hóa tốc độ cao.
* Khác biệt vượt trội của PyTorch Tensor:
  1. **Khả năng chạy trên GPU:** Tăng tốc tính toán song song hàng nghìn nhân CUDA (`tensor.to('cuda')`).
  2. **Tích hợp sẵn Autograd:** Tự động xây dựng đồ thị tính toán để đạo hàm ngược không cần con người tính tay.

### 2. Trực giác Đồ thị tính toán (Computational Graph) & Autograd
Khi bạn thiết lập `requires_grad=True` cho một tensor, PyTorch sẽ bắt đầu ghi nhớ mọi thao tác đại số tác động lên nó.
* **Forward Pass (Lan truyền xuôi):** Tính toán giá trị đầu ra từ đầu vào thông qua các phép toán nối tiếp.
* **Backward Pass (Lan truyền ngược):** Khi gọi `loss.backward()`, PyTorch sẽ tự động áp dụng **Quy tắc đạo hàm chuỗi (Chain Rule)**, đi ngược từ nút ngọn (Loss) về các nút gốc, và lưu giá trị đạo hàm riêng $\frac{\partial Loss}{\partial w}$ vào thuộc tính `w.grad`.

### 3. Tối ưu hóa Gradient Descent bằng PyTorch thuần
Với tốc độ học $\alpha$ (Learning Rate), tham số $w$ được cập nhật theo hướng ngược chiều dốc:
$$w \leftarrow w - \alpha \times \frac{\partial Loss}{\partial w}$$
*Lưu ý cốt tử:* Cần bọc bước cập nhật trọng số trong `with torch.no_grad():` để không làm ô nhiễm đồ thị tính toán!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Đồ thị tính toán tự động (Computational Graph):
     x (Input) ──┐
                 ├──> [ Phép nhân: w * x ] ──> [ Phép cộng: + b ] ──> y_hat (Dự đoán)
     w (Weight) ─┘                                                           │
                                                                             ▼
     y (Target) ───────────────────────────────────────────────────> [ Loss: MSE ]
                                                                             │
                                                                     loss.backward()
                                                                             │
    w.grad = dLoss/dw  <─────────────────────────────────────────────────────┘
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import numpy as np

# 1. Khoi tao PyTorch Tensor va quan ly thiet bi (Device Agnostic Code)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"He thong dang su dung thiet bi tinh toan: {device}")

# Tao tensor tu du lieu co san va kiem tra thuoc tinh
x_data = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=torch.float32, device=device)
print("Tensor x_data:\n", x_data)
print(f"Shape: {x_data.shape} | Dtype: {x_data.dtype} | Device: {x_data.device}")

# Chuyen doi qua lai 0-copy giua NumPy va PyTorch CPU
np_arr = np.array([10, 20, 30])
torch_from_np = torch.from_numpy(np_arr)
print("\nChuyen tu NumPy sang PyTorch:", torch_from_np)

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa thao tác cơ sở với PyTorch Tensor:
- Kỹ thuật viết code bất khả tri thiết bị (Device-agnostic) tự động nhận diện GPU nếu có.
- Khám phá các thuộc tính then chốt: `shape`, `dtype`, `device`.
- Phương thức `torch.from_numpy()` chia sẻ chung vùng nhớ bộ nhớ đệm giúp chuyển đổi không tốn chi phí copy.


In [ ]:
# 2. Co che Autograd va Vong lap Toi uu hoa tu so 0 bang Pure PyTorch
torch.manual_seed(42)

# Tao du lieu mau: y = 3.5 * x + 2.0 + nhieu
X = torch.randn(100, 1)
y_true = 3.5 * X + 2.0 + 0.2 * torch.randn(100, 1)

# Khoi tao tham so mo hinh can hoc (Weights & Bias) voi requires_grad=True
w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, 1, requires_grad=True)

learning_rate = 0.05
epochs = 50

print("===== BAT DAU TRAINING LOOP PURE PYTORCH =====")
for epoch in range(1, epochs + 1):
    # Buoc 1: Forward Pass - Tinh toan du doan
    y_pred = X @ w + b
    
    # Buoc 2: Tinh ham mat mat (MSE Loss)
    loss = torch.mean((y_pred - y_true) ** 2)
    
    # Buoc 3: Backward Pass - Tinh dao ham tu dong
    loss.backward()
    
    # Buoc 4: Cap nhat tham so bang Gradient Descent (khong luu vao do thi)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        
        # Buoc 5: Xoa trang gradient truoc luot tiep theo
        w.grad.zero_()
        b.grad.zero_()
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/{epochs} | MSE Loss: {loss.item():.4f} | w: {w.item():.3f} | b: {b.item():.3f}")

print(f"\nGia tri toi uu hoc duoc: w = {w.item():.3f} (That: 3.50), b = {b.item():.3f} (That: 2.00)")

### 🔍 Code Walkthrough (Khối 2)
Khối code tự tay viết một vòng lặp tối ưu hóa Gradient Descent thuần túy bằng PyTorch Autograd:
- `requires_grad=True`: Kích hoạt cơ chế theo dõi đạo hàm cho $w$ và $b$.
- `loss.backward()`: Tự động tính toàn bộ đạo hàm riêng.
- `with torch.no_grad()`: Tạm thời vô hiệu hóa autograd trong quá trình cập nhật trọng số.
- `w.grad.zero_()`: **Bắt buộc** xóa gradient cũ vì PyTorch mặc định sẽ cộng dồn đạo hàm (Accumulate Gradients).


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phép toán đại số với Tensors
Tạo tensor $A$ có kích thước $(3, 4)$ và tensor $B$ có kích thước $(4, 2)$ với các giá trị ngẫu nhiên. Thực hiện phép nhân ma trận $A 	imes B$ bằng toán tử `@` hoặc hàm `torch.matmul`. In ra kích thước của tensor kết quả.

### Bài 2 (Easy): Đạo hàm hàm đa thức một biến
Cho hàm số $y = 3x^3 - 5x^2 + 7x - 2$. Hãy dùng PyTorch Autograd để tính giá trị đạo hàm $\frac{dy}{dx}$ tại điểm $x = 2.0$. Đối chiếu với đạo hàm giải tích tính bằng tay ($y' = 9x^2 - 10x + 7$).

### Bài 3 (Medium): Tác động của Learning Rate
Thử nghiệm thay đổi `learning_rate` trong đoạn code trên thành $1.5$ (quá lớn) và $0.0001$ (quá nhỏ). Quan sát hiện tượng nổ mất mát (Exploding Loss / NaN) hoặc mô hình học quá chậm.

### Bài 4 (Medium): Ngắt nhánh đồ thị bằng `.detach()`
Tìm hiểu phương thức `tensor.detach()`. Viết một ví dụ nhỏ chứng minh rằng khi gọi `.detach()`, tensor mới sẽ không còn kết nối với đồ thị tính toán ban đầu và không thể lan truyền gradient ngược qua nó.

### Bài 5 (Challenge): Hồi quy phi tuyến bậc hai bằng Autograd
Tạo dữ liệu theo hàm bậc hai $y = 2x^2 - 3x + 1$. Sử dụng PyTorch Tensors với 3 tham số cần học: $w_2, w_1, b$ và viết vòng lặp tối ưu hóa để tìm lại bộ tham số trên.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các bài thi Olympic AI nâng cao, bạn thường gặp các bài toán yêu cầu thiết kế hàm mất mát tùy biến (Custom Loss Function, ví dụ Focal Loss cho phân loại mất cân bằng, hoặc Metric Learning Loss). Nếu không hiểu Autograd, bạn sẽ không biết viết hàm mất mát sao cho PyTorch có thể tự động đạo hàm được. Quy tắc sống còn: **Mọi phép toán trong Custom Loss phải dùng toán tử của PyTorch (không dùng NumPy), nếu không đồ thị tính toán sẽ bị đứt gãy và `.backward()` sẽ không hoạt động!**



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một sinh viên quên dòng lệnh xóa gradient `w.grad.zero_()`, khiến mô hình chạy qua mỗi epoch thì Loss lại tăng vọt bất thường:

```python
# CODE CÓ LỖI TÍCH LŨY GRADIENT:
# PyTorch co tinh nang mac dinh la w.grad += new_grad (Accumulate Gradients)!
# Neu quen xoa ve 0 sau moi buoc, gradient se ngay cang khong lo!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao PyTorch lại thiết kế tính năng cộng dồn gradient thay vì tự động xóa đè (gợi ý: hữu ích khi cần tích lũy gradient qua nhiều mini-batch khi GPU bị thiếu VRAM).
2. Kiểm chứng việc bỏ dòng `w.grad.zero_()` làm sai lệch kết quả tối ưu như thế nào.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Thuộc tính nào của PyTorch Tensor cần được đặt thành `True` để hệ thống tự động theo dõi và ghi lại đồ thị tính toán đạo hàm?
   - A. `is_trainable=True`
   - B. `requires_grad=True`
   - C. `track_gradient=True`
   - D. `auto_diff=True`

2. Khối lệnh `with torch.no_grad():` thường được sử dụng trong trường hợp nào?
   - A. Khi muốn tăng độ chính xác của mạng nơ-ron lên gấp đôi.
   - B. Khi cập nhật tham số hoặc khi chạy kiểm thử (Inference / Evaluation) để tiết kiệm bộ nhớ RAM/VRAM và tăng tốc độ xử lý do không cần lưu đồ thị tính toán.
   - C. Khi muốn xóa dữ liệu tập Train.
   - D. Khi chuyển code sang chạy trên GPU.

3. Tại sao sau mỗi bước cập nhật trọng số trong PyTorch, chúng ta bắt buộc phải gọi `.grad.zero_()`?
   - A. Vì nếu không xóa, PyTorch sẽ báo lỗi cú pháp.
   - B. Vì PyTorch mặc định sẽ cộng dồn gradient mới vào gradient cũ từ các vòng lặp trước, dẫn tới việc cập nhật sai lệch hướng dốc.
   - C. Để giải phóng toàn bộ card đồ họa.
   - D. Vì gradient sẽ biến thành số âm.

4. Phương thức nào dùng để chuyển một PyTorch Tensor từ CPU sang card đồ họa GPU một cách an toàn nhất?
   - A. `tensor.gpu()`
   - B. `tensor.to(device)` với `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`
   - C. `tensor.cuda_force()`
   - D. `tensor.convert_gpu()`

5. Khi ta gọi lệnh `loss.backward()`, thuật toán toán học cốt lõi nào được kích hoạt để tính toán đạo hàm ngược qua toàn bộ các tầng?
   - A. Thuật toán tìm kiếm nhị phân (Binary Search).
   - B. Quy tắc đạo hàm chuỗi hàm hợp (Chain Rule) trong giải tích vi phân.
   - C. Phép biến đổi Fourier nhanh (FFT).
   - D. Phương pháp khử Gauss-Jordan.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Bản chất của PyTorch Tensor và cơ chế quản lý phần cứng CPU/GPU.
  - Sức mạnh của Autograd và Đồ thị tính toán tự động (Computational Graph).
  - Tự tay cài đặt vòng lặp Gradient Descent thuần túy không dùng thư viện cao cấp.
* **What I can now do:**
  - Viết code tính toán đạo hàm tự động cho bất kỳ hàm mục tiêu phức tạp nào.
  - Tự tin kiểm soát và điều khiển dòng chảy của Tensor trong bộ nhớ.
* **What I still need to practice:**
  - Chuyển giao cách nạp dữ liệu thủ công sang hệ thống nạp dữ liệu theo Batch chuyên nghiệp: **Dataset & DataLoader** ở Buổi 2.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M3_S02**, chúng ta sẽ giải quyết bài toán nạp dữ liệu quy mô lớn vào bộ nhớ: **PyTorch Data Pipeline: Tùy Biến `Dataset` & `DataLoader`** — Học cách chia Mini-batch, xáo trộn dữ liệu (Shuffle), và xử lý đa luồng nạp dữ liệu!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M3**:
- PyTorch tensors & GPU device transfer ✓
- Autograd & Computational graph ✓
- Manual optimization loop via backward() & grad.zero_() ✓

* **Session Deliverable:** Mã nguồn tối ưu hóa tham số bằng PyTorch Tensor và Autograd từ số 0
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
